# LLM APIs & Application Development — Module 03 (Day 11–Day 15)
**Level:** Intermediate | **Platform:** Google Colab / Jupyter Notebook

## Learning objectives
- Understand LLM APIs, authentication and environment variables.
- Connect an LLM using Python.
- Manage system/user/assistant messages and conversation history.
- Configure model parameters and understand token usage.
- Implement structured responses and robust error handling.
- Build a chat application with custom instructions and a basic UI.

**Important:** You need your own API key and an account with API access. API usage may cost money. Never paste your key into a public notebook or upload it to GitHub.

## 1. Install dependencies

In [ ]:
!pip -q install -U openai

## 2. Configure API authentication securely
In Colab, add your key using the Secrets panel (key icon) with the secret name `OPENAI_API_KEY`. Do not write the key directly in this notebook.

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Add OPENAI_API_KEY in Colab Secrets, then rerun this cell.")

os.environ["OPENAI_API_KEY"] = api_key
os.environ.setdefault("OPENAI_MODEL", "gpt-4o-mini")
print("API key loaded securely.")

## 3. Connect to the LLM API

In [ ]:
from openai import OpenAI

client = OpenAI()  # Reads OPENAI_API_KEY from the environment
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print("Client initialized. Model configured:", MODEL)

## 4. Make your first API request

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful AI tutor."},
        {"role": "user", "content": "Explain what an LLM API is in two simple sentences."}
    ],
    max_tokens=120,
    temperature=0.4
)

print(response.choices[0].message.content)
if response.usage:
    print("Token usage:", response.usage)

## 5. Understand conversation roles and history

In [ ]:
conversation = [
    {"role": "system", "content": "You are a concise Python tutor."},
    {"role": "user", "content": "What is a Python list?"},
]
first = client.chat.completions.create(
    model=MODEL, messages=conversation, max_tokens=100, temperature=0.3
)
assistant_answer = first.choices[0].message.content
conversation.append({"role": "assistant", "content": assistant_answer})
conversation.append({"role": "user", "content": "Give me one short code example."})

second = client.chat.completions.create(
    model=MODEL, messages=conversation, max_tokens=120, temperature=0.3
)
print("Assistant:", second.choices[0].message.content)

## 6. Reusable API function with error handling

In [ ]:
from openai import OpenAI, APIError, AuthenticationError, RateLimitError

def get_ai_response(messages, model=MODEL, max_tokens=200, temperature=0.4):
    try:
        result = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature
        )
        answer = result.choices[0].message.content or ""
        usage = result.usage
        usage_info = {
            "input_tokens": usage.prompt_tokens if usage else None,
            "output_tokens": usage.completion_tokens if usage else None,
            "total_tokens": usage.total_tokens if usage else None
        }
        return {"ok": True, "answer": answer, "usage": usage_info}
    except AuthenticationError:
        return {"ok": False, "error": "Authentication failed. Check the API key."}
    except RateLimitError:
        return {"ok": False, "error": "Rate limit or quota reached. Check account limits."}
    except APIError as exc:
        return {"ok": False, "error": f"API error: {type(exc).__name__}"}
    except Exception as exc:
        return {"ok": False, "error": f"Unexpected error: {type(exc).__name__}"}

test = get_ai_response([{"role": "user", "content": "Say hello in one sentence."}])
print(test)

## 7. Structured response (JSON)
Ask for a JSON object and validate it. Model output is not guaranteed to be valid JSON, so always parse and handle errors.

In [ ]:
import json

json_messages = [
    {"role": "system", "content": "Return only a valid JSON object. Do not use Markdown fences."},
    {"role": "user", "content": 'Create a task object with keys "title", "priority", and "completed". Task: Revise API concepts. Use a string, a priority string, and a boolean.'}
]
result = get_ai_response(json_messages, max_tokens=120)
print(result)

if result["ok"]:
    try:
        task_data = json.loads(result["answer"])
        print("Validated JSON:", task_data)
    except json.JSONDecodeError:
        print("The response was not valid JSON. Review it before using it.")

## 8. Token usage and basic cost management
The API returns token usage for many requests. Actual cost depends on the model, input/output token rates, cached tokens and provider pricing. Do not assume a fixed price; check the provider's current pricing page and your account usage.

In [ ]:
if "response" in globals() and response.usage:
    print("Input tokens:", response.usage.prompt_tokens)
    print("Output tokens:", response.usage.completion_tokens)
    print("Total tokens:", response.usage.total_tokens)

# Practical ways to manage usage:
# 1. Set a reasonable max_tokens value.
# 2. Keep only relevant recent conversation history.
# 3. Choose a model that fits the task and budget.
# 4. Monitor usage in your provider dashboard.

## 9. Build the chat application
The complete Streamlit application is provided in the GitHub repository as `app.py`. It includes user input, assistant responses, session history, custom instructions, parameter controls, token usage and basic error handling.

In [ ]:
# To run locally after downloading/cloning the repository:
# 1. Install: pip install -r requirements.txt
# 2. Create .env from .env.example and add your API key.
# 3. In a terminal, run:
#    streamlit run app.py
#
# For Colab, a simple command-line chat loop can demonstrate conversation history:
chat_history = [
    {"role": "system", "content": "You are a helpful AI assistant."}
]
print("Type 'exit' to stop.")
while True:
    user_text = input("You: ")
    if user_text.strip().lower() == "exit":
        break
    chat_history.append({"role": "user", "content": user_text})
    reply = get_ai_response(chat_history, max_tokens=250)
    if reply["ok"]:
        print("Assistant:", reply["answer"])
        chat_history.append({"role": "assistant", "content": reply["answer"]})
        print("Usage:", reply["usage"])
    else:
        print("Error:", reply["error"])

## 10. Testing checklist
After running the app, test these cases and record the actual result:

| Test | Expected behaviour |
|---|---|
| Normal question | Assistant returns a relevant response |
| Follow-up question | Assistant uses earlier conversation context |
| Custom instruction | Response follows the selected system instruction |
| Clear conversation | Chat history is cleared |
| Missing API key | App displays a setup message |
| Invalid key / quota issue | App displays a safe error message |
| Long response | Response respects the configured output-token limit |

**Testing report**
- Test date:
- Model configured:
- Successful tests:
- Errors observed:
- Token usage observed:
- Improvements planned:

## 11. Deployment: publish a live website
1. Push the repository to GitHub.
2. Create a Streamlit Community Cloud app connected to the repository and select `app.py`.
3. Add `OPENAI_API_KEY` and `OPENAI_MODEL` in deployment Secrets.
4. Deploy and test the generated public URL.

Never put the API key in GitHub or in frontend code. Add authentication and rate limiting before making a production app available to everyone.

## 12. Conclusion
In this module, I learned how to authenticate with an LLM API using environment variables, send messages through Python, manage system/user/assistant conversation history, configure model parameters, inspect token usage, validate structured JSON responses and handle common API errors. I also built a Streamlit chat application with user input, AI responses, conversation history and custom instructions.

**Submission note:** Run the API notebook using your own key, test the app, publish the GitHub repository, deploy the Streamlit app, and add the actual live website URL to your submission.